**This script traines a polynomial regression and generative additive model on generic turbine data sets, predicting DELs form environmental conditions**

In [ ]:
#Importing libraries
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold
from sklearn.metrics import make_scorer
from sklearn.model_selection import cross_validate

from pygam import LinearGAM, s, te

from lifelines.utils import concordance_index

import time

import joblib


**Data preprocessing**

In [ ]:
GT_name = 'GT10' #this is the GT to be modelled
my_path = f'add_path_here' #path leading to the GT file
GT_file = f'{GT_name}_DB_list.csv'

GT_df = pd.read_csv(f'{my_path}{GT_file}') #File name of GT file

#environmental predictors
env_inputs = [
    'windSpeed',
    'iRef',
    'shearExp',
    'density',
    'inFlowAngle'
]

#variable to be predicted
output = 'damage'

#node/load/wohler subsets to be included
nodes = ['blade_root', 'tower_base']
loads = ['Mx', 'My']
wohlers = [4, 9, 10, 14]

#exluding these nodes because they are not essential for analysis
excluded_nodes = {'stationary_hub', 'rotating_hub', 'blade_maxchord', 'tower_top'}
GT_df = GT_df[~GT_df['node'].isin(excluded_nodes)]

#decide whether the GT data will be DEL normalised
damage_norm = False


**Damage Normalisation**

Normalising the damage to see if the damage relationships are the same across GTs to potentially reduce error.

In [ ]:
#------------------------------------------------------damage normalisation---------------------------------------------------------------------
#baseline values
windspeed_base = 16 #found to be optimum
iref_base = 0.14
density_base = 1.15
inflow_base = 0
shear_base = 0.15

#averaging across turbulence seeds for the damage does not work as they need to be raised to the power of the wohler exp first.
def power_average_damage(damage_values, wohler):
    return (np.mean(damage_values ** wohler)) ** (1/wohler)


#finding the damage value that matches the baseline environmental conditions
#this creates a df with a corresponding baseline damage for each subset
baseline_rows = GT_df.loc[
    (GT_df['windSpeed'] == windspeed_base) &
    (GT_df['iRef'] == iref_base) &
    (GT_df['density'] == density_base) &
    (GT_df['shearExp'] == shear_base) &
    (GT_df['inFlowAngle'] == inflow_base),
    ['node', 'load', 'wohler', 'damage']
]

# apply power average across the three seeds for each subset
# the lambda x creates another function to handle each 3 identical rows seperately, where x is each group
baseline = baseline_rows.groupby(['node', 'load', 'wohler']).apply(
    lambda x: pd.Series({
        'baseline_damage': power_average_damage(x['damage'].values, x.name[2])
    }) ,
    include_groups=False
).reset_index()

# merge 
GT_df = GT_df.merge(baseline, on=['node', 'load', 'wohler'], how='left')

#only applied if set to normalise
if damage_norm:
    GT_df['damage'] = GT_df['damage'] / GT_df['baseline_damage']


**Polynomial Regression Model**
Uses polynomial feature expansion to createpolynomial features from input variables then performs linear regression on the features

In [ ]:
#a function to find the c-index via cross validation scoring
def cindex_scorer(y_true, y_pred):
    return concordance_index(y_true, y_pred)
cindex_score = make_scorer(cindex_scorer, greater_is_better=True)

poly_reg_results = pd.DataFrame()
cv_results = pd.DataFrame()
poly_pdep = {} #for partial dependence plot

#looping through all node/load/wohler combinations
for node in nodes:
    for load in loads:
        for wohler in wohlers:
            subset = GT_df[
                (GT_df['node'] == node) & 
                (GT_df['load'] == load) & 
                (GT_df['wohler'] == wohler)
            ]
            if len(subset) == 0:
                continue

            #dependent and independent variables
            X = subset[env_inputs]
            y = subset[output]

            #trying to 2,3 and 4 degrees to see if more complexity is helpful
            for degree in [3]:

#------------------------------------------------------------------FITTING AND PREDICTING-------------------------------------------------------------------------------------
                #not using cross validation gere
                reg_model = Pipeline([('scaler', StandardScaler()), ('poly', PolynomialFeatures(degree=degree, include_bias=False)), ('regression', LinearRegression())])
                reg_model.fit(X, y)

#------------------------------------------------------------------FOR DEPENDENCY PLOTS---------------------------------------------------------------------------------------

                for i, var in enumerate(env_inputs):
                                    x_range = np.linspace(X[var].min(), X[var].max(), 100)
                                    X_grid = pd.DataFrame(np.tile(X.mean().values, (100, 1)), columns=env_inputs)
                                    X_grid[var] = x_range
                                    y_poly = reg_model.predict(X_grid)
                                    poly_pdep[(node, load, wohler, var)] = (x_range, y_poly)
            
#------------------------------------------------------------------CROSS VALIDATION AND SCORING -------------------------------------------------------------------------------------------------
                #also trying cross validation scoring
                scores = cross_val_score(reg_model, X, y, cv=5, scoring='r2')
                cv_results.loc[f'{node}_{load}_{wohler}', f'degree_{degree}_mean_r2'] = f'{scores.mean():.3f}'
                cv_results.loc[f'{node}_{load}_{wohler}', f'degree_{degree}_std'] = f'{scores.std():.3f}'
                # RMSE via cross validation
                mse_scores = cross_val_score(reg_model, X, y, cv=5, scoring='neg_mean_squared_error')
                mean_rmse = np.sqrt(-mse_scores.mean())
                # MAE via cross validation
                mae_scores = cross_val_score(reg_model, X, y, cv=5, scoring='neg_mean_absolute_error')
                mean_mae = -mae_scores.mean()
                #getting the percentage error
                rmse_pct = (mean_rmse / y.mean()) * 100
                mae_pct = (mean_mae / y.mean()) * 100

                #calculating c-index
                c_index = cross_val_score(reg_model, X, y, cv=5, scoring=cindex_score)
                mean_cdx = c_index.mean()

                #cross validating
                cv_results2 = cross_validate(
                reg_model, X, y, cv=5,
                scoring=['r2', 'neg_root_mean_squared_error'],
                return_train_score=True
                )

                train_r2 = cv_results2['train_r2']
                test_r2 = cv_results2['test_r2']
                train_rmse = -cv_results2['train_neg_root_mean_squared_error']
                test_rmse = -cv_results2['test_neg_root_mean_squared_error']

                # Putting the rmse and mae scoring in the results df
                cv_results.loc[f'{node}_{load}_{wohler}', f'degree_{degree}_rmse'] = round(mean_rmse, 3)
                cv_results.loc[f'{node}_{load}_{wohler}', f'degree_{degree}_rmse_pct'] = round(rmse_pct, 1)
                cv_results.loc[f'{node}_{load}_{wohler}', f'degree_{degree}_mae'] = round(mean_mae, 3)
                cv_results.loc[f'{node}_{load}_{wohler}', f'degree_{degree}_mae_pct'] = round(mae_pct, 1)
                cv_results.loc[f'{node}_{load}_{wohler}', f'c_index'] = round(mean_cdx, 3)

                cv_results.loc[f'{node}_{load}_{wohler}', 'fit_time_s'] = round(cv_results2['fit_time'].mean(), 4)
                cv_results.loc[f'{node}_{load}_{wohler}', 'predict_time_s'] = round(cv_results2['score_time'].mean(), 4)
                #putting the results in a csv file
                cv_results.to_csv(f'{my_path}poly_exp_regression_cv_results_deg{degree}.csv')  
            
                #trying to see the coefficients for each feature
                poly = reg_model.named_steps['poly']
                feature_names = poly.get_feature_names_out(env_inputs)

#------------------------------------------------------------------COEFFICIENT EXTRACTION -------------------------------------------------------------------------------------------------
                # get coefficients
                coefficients = reg_model.named_steps['regression'].coef_
                coef_df = pd.DataFrame({
                    'feature': feature_names,
                    'coefficient': coefficients
                }).sort_values('coefficient', ascending=False)
                #coef_df.to_csv(f'{my_path}/Regression/PolyExp/poly_expdeg{degree}_regression_coeffs_{node}_{load}_{wohler}.csv')

                #this is the non-cross validation score
                score = reg_model.score(X, y)
                poly_reg_results.loc[f'{node}_{load}_{wohler}', degree] = round(score, 3) 

#------------------------------------------------------------------SAVING TRAINED MODEL -------------------------------------------------------------------------------------------------
                # #saving the trained model
                if damage_norm:
                     joblib.dump(reg_model, f'{my_path}/poly_model_{node}_{load}_{wohler}_normed.pkl')
                else:
                      joblib.dump(reg_model, f'{my_path}/poly_model_{node}_{load}_{wohler}.pkl')  
                
print(cv_results)



**General Additive Model (GAM)**

In [ ]:
gam_results = pd.DataFrame()
gam_pdep = {} #for dependency plot
gam_pdep_manual = {}

#looping through all node/load/wohler combinations
for node in nodes:
    for load in loads:
        for wohler in wohlers:
            subset = GT_df[
                (GT_df['node'] == node) & 
                (GT_df['load'] == load) & 
                (GT_df['wohler'] == wohler)
            ]
            if len(subset) == 0:
                continue

            X = subset[env_inputs].values
            y = subset[output].values

#---------------------------------------------------------- MANUALLY DOING CROSS VALIDATION ----------------------------------------------------------------------------
            kf = KFold(n_splits=5, shuffle=True, random_state=42) # have to implement cross validation manually due to some environment issues.
            fold_scores = []
            fold_mse = []
            fold_mae = []
            fold_c_index = []
            all_y_test = []
            all_y_pred = []
            all_wind_speeds = []
            all_iref = []
            all_env_values = {var: [] for var in env_inputs}

            train_r2_scores, test_r2_scores = [], []
            train_rmse_scores, test_rmse_scores = [], []     

            #assessing the difference between the fit and predict times for both models
            fit_times = []         
            predict_times = [] 

            #cross val
            for train_idx, test_idx in kf.split(X):
                X_train, X_test = X[train_idx], X[test_idx]
                y_train, y_test = y[train_idx], y[test_idx]

                #this is dependent on an X order of wind speed, iref, shear exp, density, inflow angle
                gam = LinearGAM(    s(0, n_splines=10) +   # windSpeed - 10 unique values
                                    s(1, n_splines=7) +   # iRef - 7 unique values
                                    s(2, n_splines=4) +   # shearExp - 4 unique values
                                    s(3, n_splines=4) +   # density - 4 unique values
                                    s(4, n_splines=4) +   # inflow angle 3 unique but need 4
                                    te(0,1)+te(0,2)+te(0,4)+te(1,3)+te(0,3))

                start_fit = time.time()
                gam.fit(X_train, y_train)   
                fit_times.append(time.time() - start_fit)

                start_pred = time.time()
                y_pred = gam.predict(X_test)
                predict_times.append(time.time() - start_pred)
#------------------------------------------------------------------SCORING -------------------------------------------------------------------------------------------------
                y_train_pred = gam.predict(X_train)
                train_r2_scores.append(r2_score(y_train, y_train_pred))
                train_rmse_scores.append(np.sqrt(mean_squared_error(y_train, y_train_pred)))

                ss_res = np.sum((y_test - y_pred) ** 2)
                ss_tot = np.sum((y_test - np.mean(y_test)) ** 2)
                r2 = 1 - ss_res / ss_tot
                fold_scores.append(r2)
                fold_mse.append(mean_squared_error(y_test, y_pred)) #getting the MSE score
                fold_mae.append(mean_absolute_error(y_test, y_pred)) #gettign the MAE score
                fold_c_index.append(concordance_index(y_test, y_pred)) #getting concordance index wihtin folds

                #collecting predictions across folds
                all_y_test.extend(y_test)
                all_y_pred.extend(y_pred)

                for i, var in enumerate(env_inputs):
                    all_env_values[var].extend(X_test[:, i])

                #for residual scatter colouring
                all_wind_speeds.extend(X_test[:, 0])
                all_iref.extend(X_test[:, 1])

            train_r2_mean = np.mean(train_r2_scores)
            test_r2_mean = np.mean(fold_scores)  
            r2_gap = train_r2_mean - test_r2_mean

            #constructing the scoring dataframes
            gam_results.loc[f'{node}_{load}_{wohler}', 'mean_r2'] = round(np.mean(fold_scores), 3)
            gam_results.loc[f'{node}_{load}_{wohler}', 'std'] = round(np.std(fold_scores), 3)
            gam_results.loc[f'{node}_{load}_{wohler}', 'train_r2'] = round(train_r2_mean, 3)
            gam_results.loc[f'{node}_{load}_{wohler}', 'r2_gap'] = round(r2_gap, 3)
            mean_rmse = np.sqrt(np.mean(fold_mse))
            mean_mae = np.mean(fold_mae)
            mean_cidx = np.mean(fold_c_index)
            rmse_pct = (mean_rmse / y.mean()) * 100
            mae_pct = (mean_mae / y.mean()) * 100
            gam_results.loc[f'{node}_{load}_{wohler}', 'rmse'] = round(mean_rmse, 3)
            gam_results.loc[f'{node}_{load}_{wohler}', 'rmse_pct'] = round(rmse_pct, 1)
            gam_results.loc[f'{node}_{load}_{wohler}', 'mae'] = round(mean_mae, 3)
            gam_results.loc[f'{node}_{load}_{wohler}', 'mae_pct'] = round(mae_pct, 1)
            gam_results.loc[f'{node}_{load}_{wohler}', 'c_index'] = round(mean_cidx, 3)

            gam_results.loc[f'{node}_{load}_{wohler}', 'fit_time_s'] = round(np.mean(fit_times), 4)
            gam_results.loc[f'{node}_{load}_{wohler}', 'predict_time_s'] = round(np.mean(predict_times), 4) 

#---------------------------------------------------------------non-cross validates -------------------------------------------------------------------------------------
         
            gam_full = LinearGAM(s(0) + s(1) + s(2) + s(3) + s(4) + te(0,1) + te(0,3)+ te(1,3)+ te(0,4)+ te(0,2))
            gam_full.fit(X, y)

#--------------------------------------------------------------FOR DEPENDENCY PLOT-------------------------------------------------------------------------------------------------
            
            # uses gam full
            for i, var in enumerate(env_inputs):
                x_range = np.linspace(X[:, i].min(), X[:, i].max(), 100)
                X_grid = np.tile(X.mean(axis=0), (100, 1))
                X_grid[:, i] = x_range
                y_gam = gam_full.predict(X_grid)   # CHANGED: gam -> gam_full
                gam_pdep_manual[(node, load, wohler, var)] = (x_range, y_gam)
            # for i, var in enumerate(env_inputs):
            #     XX = gam_full.generate_X_grid(term=i)
            #     pdep, confi = gam_full.partial_dependence(term=i, X=XX, width=0.95)
            #     gam_pdep[(node, load, wohler, var)] = (XX[:, i], pdep, confi)

#----------------------------------------------------------SAVING MODEL------------------------------------------------------------------------------------------------------------
            if damage_norm:
                    joblib.dump(gam_full, f'{my_path}/gam_model_{node}_{load}_{wohler}_normed.pkl')
            else:
                    joblib.dump(gam_full, f'{my_path}/gam_model_{node}_{load}_{wohler}.pkl')  

#saving result csv files
if damage_norm:
    gam_results.to_csv(f'{my_path}{GT_name}_gam_result_with_time.csv')
else:
    gam_results.to_csv(f'{my_path}{GT_name}_gam_result_report.csv')

**plotting comparisons in performance**

In [ ]:
#---------------------------------------------------------------------PLOTTING GAM VS PRM predictions--------------------------------------------------------------------------

gam_color = "#F77F54CD"
pfe_color = "#2C706B" 

class OneDecimalFormatter(ScalarFormatter):
    def _set_format(self):
        self.format = '%.1f'


import matplotlib as mpl
mpl.rcParams['font.size'] = 26
mpl.rcParams['axes.titlesize'] = 28
mpl.rcParams['axes.labelsize'] = 28
mpl.rcParams['xtick.labelsize'] = 24
mpl.rcParams['ytick.labelsize'] = 24
mpl.rcParams['legend.fontsize'] = 24
mpl.rcParams['figure.dpi'] = 600

var_labels = {
    'windSpeed': 'Wind speed (m/s)',
    'iRef': 'Turbulence intensity',
    'shearExp': 'Shear exponent',
    'inFlowAngle': 'Inflow angle (°)',
    'density': 'Air density (kg/m³)'
}
subset_count = 0

for node in nodes:
    for load in loads:
        for wohler in wohlers:
            if (node, load, wohler, env_inputs[0]) not in poly_pdep:
                continue
            if f'{node}_{load}_{wohler}' not in ['blade_root_Mx_14', 'blade_root_My_14', 'tower_base_My_9']:
                continue
            fig, axes = plt.subplots(1, len(env_inputs), figsize=(19, 4.3))
            for i, (ax, var) in enumerate(zip(axes, env_inputs)):
                x_poly, y_poly = poly_pdep[(node, load, wohler, var)]
                x_gam, pdep = gam_pdep_manual[(node, load, wohler, var)]
                ax.plot(x_poly, y_poly, label='PRM', color=pfe_color)
                ax.plot(x_gam, pdep, label='GAM', color=gam_color)
                ax.set_xlabel(var_labels.get(var, var))

                formatter = OneDecimalFormatter(useMathText=True)
                formatter.set_powerlimits((0, 0))
                ax.yaxis.set_major_formatter(formatter)

                if i == 0:
                    ax.set_ylabel('Predicted DEL (Nm)')
                    if subset_count == 0:
                        ax.legend(loc = 'upper left', frameon=False, bbox_to_anchor=(-0.08, 1.1))
                    else:
                        ax.legend(loc = 'lower right', frameon=False, bbox_to_anchor=(1.1, -0.1))
            plt.tight_layout()
            plt.savefig(f'{my_path}dep_comp_{node}_{load}_{wohler}.pdf')
            subset_count += 1 


In [ ]:
#------------------------------------------------------------------------------------- PLOTTING C-INDEX AND RMSE-------------------------------------------------------------------------------------------
#setting plot appearance
import matplotlib as mpl
mpl.rcParams['font.size'] = 10
mpl.rcParams['axes.titlesize'] = 10
mpl.rcParams['axes.labelsize'] = 10
mpl.rcParams['xtick.labelsize'] = 10
mpl.rcParams['ytick.labelsize'] = 10
mpl.rcParams['legend.fontsize'] = 10
mpl.rcParams['figure.dpi'] = 600

plt.rcParams['font.family'] = 'STIXGeneral'
plt.rcParams['mathtext.fontset'] = 'stix'

metric = ['% RMSE']
subsets = ['Blade \n root \n Mx 4', 'Blade \n  root \n Mx 9', 'Blade \n root \n Mx 10', 'Blade \n root \n Mx 14','Blade \n root \n My 4', 'Blade \n root \n My 9', 'Blade \n root \n My 10', 'Blade \n root \n My 14', 'Tower \n base \n My 4', 'Tower \n base \n My 9']

#determined from csv files
pfe_values = [1.4, 1.4, 1.5, 1.8, 2.1, 3.9, 4.3, 5.5, 6.5, 10.4]   # PFE results
gam_values = [0.9, 1.2, 1.3, 1.7, 1.8, 3.5, 3.9, 4.9, 5.5, 9.6]  # GAM results

pfe_values_c = [0.947, 0.952, 0.953, 0.953, 0.982, 0.965, 0.961, 0.950, 0.955, 0.916]   # PFE results
gam_values_c = [0.974, 0.969, 0.968, 0.964, 0.985, 0.969, 0.966, 0.956, 0.965, 0.931]  # GAM results

x = np.arange(len(subsets))  # label locations
width = 0.35  # width of each bar

gam_color = "#F77F54CD"
pfe_color = "#2C706B" 

#bar plot for RNSE
fig, ax = plt.subplots(figsize=(6.27, 3.5))
bars1 = ax.bar(x - width/2, pfe_values, width, label='PRM', color = pfe_color )
bars2 = ax.bar(x + width/2, gam_values, width, label='GAM', color = gam_color)

ax.set_ylabel('% RMSE')
ax.set_xticks(x)
ax.set_xticklabels(subsets)
ax.legend(frameon = False)

plt.tight_layout()
plt.savefig(f'{my_path}model_comparison.pdf')
plt.show()

#dot plot for c-index
fig, ax = plt.subplots(figsize=(6.27, 3.5))
ax.plot(x + 0.1, pfe_values_c, 'o', color=pfe_color, label='PRM', markersize=6)
ax.plot(x - 0.1, gam_values_c, 'o', color=gam_color, label='GAM', markersize=6)
for i in range(len(subsets)):
    ax.plot([i - 0.1, i + 0.1], [gam_values_c[i], pfe_values_c[i]],
            color='gray', linewidth=0.8, alpha=0.5)
ax.set_ylabel('C-index')
ax.set_xticks(x)
ax.set_xticklabels(subsets)
ax.set_ylim(0.90, 1.0)
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(f'{my_path}model_comparison_c_index.pdf')
plt.show()
